# 00 — Present với cô Châu (31/10/2026)

Việc 22 (plan §12). Notebook này là bản trình bày: dàn ý 10 phần trong `docs/plan.md` §12, thêm một phần
"KGAT và T-KGAT (sơ bộ)" ở cuối. Mỗi phần ghi rõ người trình bày.

**Quy tắc của notebook**
- **Không huấn luyện lại.** Chỉ đọc kết quả đã lưu và vẽ hình từ `docs/figures/` hoặc từ dữ liệu đã xử lý.
- **Không gõ số bằng tay.** Mọi con số trong lời dẫn được in ra từ code: từ `results/registry.csv` (qua
  `tkgat.registry.load()`) hoặc từ `docs/data-report-numbers.md` (do `scripts/07_report_numbers.py` sinh). Chỗ nào
  chưa có số thì ghi "chưa có" kèm lý do.
- **Điều kiện chạy:** đã dựng `data/processed/` (scripts 02, 03, 06) như trong `CLAUDE.md`; "Restart and run all" không lỗi,
  dưới 2 phút.

| # | Phần | Người trình bày |
|---|---|---|
| 1 | Góp ý ngày 17/10 và việc đã làm | Phôn |
| 2 | Vì sao làm lại: 3 lỗi rò rỉ của kết quả cũ | Triết |
| 3 | Định hướng ngành Hệ thống thông tin và 3 câu hỏi nghiên cứu | Phôn |
| 4 | MovieLens-100K theo thời gian: phiên nhập rating, hệ quả cho cách chia và cho Δt | Triết |
| 5 | Liên kết KB4Rec | Phôn |
| 6 | KG theo thời gian | Triết |
| 7 | Quy trình chia train/val/test và bằng chứng không rò rỉ | Triết |
| 8 | Baseline trên 100K và 1M | Phôn |
| 9 | Repo và cách làm việc mới | Phôn |
| 10 | Kế hoạch từ 02/11 và câu hỏi xin ý kiến cô | Cả hai |
| 11 | KGAT và T-KGAT (sơ bộ) | Triết |

In [ ]:
import ast
import dataclasses
import inspect
import re
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from tkgat import registry
from tkgat.analysis.report import parse_numbers
from tkgat.data.leakage import check_split
from tkgat.pipeline.build import load_config, load_saved
from tkgat.train.bpr import fit_bpr

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
NUMS = parse_numbers((ROOT / "docs/data-report-numbers.md").read_text(encoding="utf-8"))
REG = registry.load(ROOT / "results/registry.csv")  # newest run per dataset/split/model/config/seed/phase
HAVE_DATA = all((ROOT / "data/processed" / d / "meta.json").exists() for d in ("ml-100k", "ml-1m"))
LABEL = {"ml100k": "ML-100K", "ml1m": "ML-1M"}
MODEL_NAME = {"mostpop": "MostPop", "mostpop_recent": "MostPop recent", "bprmf": "BPR-MF"}


def V(key):
    # a number or text of docs/data-report-numbers.md, exactly as the script wrote it (Vietnamese format)
    if key not in NUMS:
        raise KeyError(f"{key!r} is not in docs/data-report-numbers.md")
    return NUMS[key][0]


def say(text):
    display(Markdown(text))


def figure(name, width=760):
    path = ROOT / "docs/figures" / name
    if path.exists():
        display(Image(filename=path, width=width))
    else:
        say(f"*chưa có: `docs/figures/{name}` (chạy `python scripts/07_report_numbers.py`).*")


def read_text(rel):
    return (ROOT / rel).read_text(encoding="utf-8")


def test_functions(pattern="test_"):
    # names of the test functions in tests/*.py that start with ``pattern``, read from the source files
    names = []
    for path in sorted((ROOT / "tests").glob("test_*.py")):
        for node in ast.walk(ast.parse(path.read_text(encoding="utf-8"))):
            if isinstance(node, ast.FunctionDef) and node.name.startswith(pattern):
                names.append((path.name, node.name))
    return names


say(
    f"> Số liệu đọc từ `docs/data-report-numbers.md` ({len(NUMS)} khóa) và `results/registry.csv` "
    f"({len(REG)} dòng mới nhất mỗi khóa). Wikidata là **bản chụp ngày {V('kg.snapshot')}**, không phải tri thức "
    "có sẵn vào thời điểm của các rating: mọi kết quả dùng KG phải nêu điều này."
)
if not HAVE_DATA:
    say('**Chưa có `data/processed/`:** các ô kiểm tra trên split sẽ báo "chưa có"; chạy scripts 02, 03, 06 trước.')

## 1. Góp ý ngày 17/10 và việc đã làm cho từng góp ý
*Người trình bày: Phôn*

Ba góp ý của cô (biên bản `docs/meetings/Meeting_Notes_17102026.md`): thiếu tập validation (nên có rò rỉ), chưa khám phá
dữ liệu đủ kỹ, Knowledge Graph chưa tôn trọng chiều thời gian. *(Phôn bổ sung lời dẫn và đối chiếu từng góp ý với việc đã làm.)*

Các quyết định đã ghi lại (đọc từ `docs/decisions/`):

In [ ]:
rows = []
for path in sorted((ROOT / "docs/decisions").glob("0*.md")):
    lines = path.read_text(encoding="utf-8").splitlines()
    status = next((x for x in lines if x.startswith("- Status:")), "- Status: ?")
    rows.append(
        {
            "quyết định": path.stem,
            "tiêu đề": lines[0].lstrip("# ").strip(),
            "trạng thái": status.removeprefix("- Status:").strip().split(" (")[0].split(".")[0],
        }
    )
display(pd.DataFrame(rows))

## 2. Vì sao làm lại: 3 lỗi rò rỉ của kết quả cũ
*Người trình bày: Triết*

Ngày 17/10 nhóm xác định kết quả của các giai đoạn trước không dùng tiếp được vì cùng mắc ba lỗi rò rỉ
(`docs/meetings/Meeting_Notes_17102026.md`, quyết định 0001). Bảng dưới đối chiếu từng lỗi với cách chặn hiện tại và
bằng chứng có thể kiểm lại ngay trong notebook. **Số liệu của kết quả cũ không nằm trong repo này (repo cũ chỉ để lưu trữ),
nên notebook không trích số nào của chúng: chưa có, và cũng không cần để lập luận.**

In [ ]:
fit_params = list(inspect.signature(fit_bpr).parameters)
test_params = [p for p in fit_params if "test" in p] or "không có"
kcore_tests = [name for file, name in test_functions("test_kcore") if file == "test_split.py"]
rows = [
    {
        "lỗi của kết quả cũ": "Không có tập validation: epoch tốt nhất được chọn trên test",
        "quy trình mới": "Có tập val; epoch, siêu tham số, biến thể chọn theo val; test chấm một lần, sau khi chọn",
        "bằng chứng kiểm lại được": f"fit_bpr nhận các tham số {fit_params}; tham số có chữ test: {test_params}",
    },
    {
        "lỗi của kết quả cũ": "K-core trên toàn bộ dữ liệu trước khi chia, nên dùng cả tương tác của giai đoạn test",
        "quy trình mới": "K-core chỉ tính trên train, sau khi chia theo mốc",
        "bằng chứng kiểm lại được": (
            f"test {kcore_tests} (tests/test_split.py); check_split báo {V('ml100k.check_split_problems')} vấn đề "
            f"trên ML-100K và {V('ml1m.check_split_problems')} trên ML-1M"
        ),
    },
    {
        "lỗi của kết quả cũ": "KG dựng từ metadata hiện tại cho các tương tác trong quá khứ",
        "quy trình mới": "KG theo mốc: train/val dùng KG tại t1, test dùng KG tại t2; chỉ giữ fact đã đúng trước mốc",
        "bằng chứng kiểm lại được": f"Titanic: {V('titanic.missing_t1')} fact bị loại tại t1 (phần 6)",
    },
]
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(rows).style.hide(axis="index"))

## 3. Định hướng theo ngành Hệ thống thông tin và 3 câu hỏi nghiên cứu
*Người trình bày: Phôn*

*(Phôn bổ sung cách đặt vấn đề theo ngành Hệ thống thông tin: tích hợp và quản lý dữ liệu cho hệ gợi ý, T-KGAT là mô hình
dùng để đánh giá giá trị của dữ liệu.)* Ba câu hỏi nghiên cứu, đọc từ `docs/plan.md`:

In [ ]:
rq = [
    cell.strip()
    for line in read_text("docs/plan.md").splitlines()
    if line.startswith("| RQ")
    for cell in [line.split("|")[1]]
]
say("\n".join(f"- {x}" for x in rq) if rq else "*chưa có: không tìm thấy dòng RQ trong `docs/plan.md`.*")

## 4. MovieLens-100K theo thời gian: rating được nhập theo phiên, hệ quả cho cách chia và cho attention theo Δt
*Người trình bày: Triết*

**Timestamp** của MovieLens, nhóm *hiểu* là lúc user ghi rating. Đây là giả định của nhóm: README của GroupLens chỉ nói
đó là số giây tính từ epoch (quyết định 0008). Giả định khớp với dữ liệu, như các số dưới đây cho thấy.

In [ ]:
say(
    f"""
- **Rating theo phiên.** {V("ml100k.first_24h_share")} rating của ML-100K ({V("ml1m.first_24h_share")} với
  ML-1M) nằm trong 24 giờ đầu kể từ rating đầu tiên của user. Một phiên nhập rating (nghỉ quá 60 phút thì
  kết thúc) có độ dài trung vị {V("ml100k.session_median_minutes")} phút ở ML-100K
  ({V("ml1m.session_median_minutes")} phút ở ML-1M).
- **Δt giữa các cạnh của một user gần bằng 0.** {V("ml100k.user_spread_under_1h")} user của ML-100K
  ({V("ml1m.user_spread_under_1h")} ở ML-1M) có toàn bộ cạnh train trải trong chưa đầy 1 giờ. Với một
  **item** thì khác: các cạnh vào item đến từ nhiều user ở nhiều thời điểm, độ trải trung vị là
  {V("ml100k.item_std_days_median")} ngày ({V("ml1m.item_std_days_median")} ngày ở ML-1M).
"""
)
figure("fig1_rating_theo_tuan.png")
figure("fig2_do_trai_thoi_gian.png")

**Hệ quả cho cách chia.** Vì phần lớn user nhập rating trong vài giờ đầu, chia theo mốc chung t1/t2 làm phần val và test
chủ yếu là user mới, chưa có trong train. Quy trình loại họ (không có thông tin để gợi ý) và đếm lại số bị loại:

In [ ]:
say(
    f"""
- **Val** ({V("ml100k.val_window_days")} ngày sau t1 = {V("ml100k.t1")}):
  {V("ml100k.val_positives_before_filter")} positive, {V("ml100k.val_dropped_cold_user")} bị loại vì user
  chưa có trong train; còn {V("ml100k.val_positives")} positive của {V("ml100k.val_users")} user.
- **Test** ({V("ml100k.test_window_days")} ngày từ t2 = {V("ml100k.t2")}):
  {V("ml100k.test_positives_before_filter")} positive, {V("ml100k.test_dropped_cold_user")} bị loại vì user
  chưa có trong train ({V("ml100k.test_cold_users")} user bị loại, {V("ml100k.test_cold_users_new")} trong
  số đó là user mới); còn {V("ml100k.test_positives")} positive của **{V("ml100k.test_users")} user**. Đây
  là lý do nhóm chạy song song trên ML-1M ({V("ml1m.test_users")} user test).
"""
)

**Hệ quả cho attention theo Δt** (quyết định 0005, chốt chi tiết ở quyết định 0009): trừ Δt vào logit attention của cạnh
user–item **trước** softmax,

$$\pi_{\text{time}}(h,r,t) = \pi(h,r,t) - \gamma\,\delta_r\,\Delta t_{\text{norm}}, \qquad \gamma = 0 \Rightarrow \text{KGAT}.$$

- Ở **phía user**, nếu mọi cạnh của một user có Δt như nhau thì phép trừ triệt tiêu trong softmax, nên suy giảm không có tác dụng.
- Ở **phía item**, các cạnh vào item có Δt khác nhau (số liệu trên), nên suy giảm vẫn có thể có tác dụng.
- **Đối chứng bắt buộc:** hoán vị timestamp của các cạnh train rồi chạy lại. Nếu kết quả không đổi thì cải thiện không đến từ
  thông tin thời gian. Một kết quả "không cải thiện" trên 100K là kết quả hợp lệ.

## 5. Liên kết KB4Rec: độ phủ, kết quả kiểm tra tay, lỗi phát hiện
*Người trình bày: Phôn*

KB4Rec chỉ liên kết MovieLens-20M. ML-1M dùng chung id với ML-20M; ML-100K được ánh xạ sang ML-1M theo tiêu đề và năm
(quyết định 0003). *(Phôn bổ sung lời dẫn.)*

In [ ]:
say(
    f"""
- **Độ phủ** (không phải độ đúng): {V("ml100k.link_freebase_items")} trong {V("ml100k.link_catalog_items")}
  item của ML-100K ({V("ml100k.link_freebase_item_share")}) có liên kết Freebase;
  {V("ml100k.link_freebase_rating_share")} rating thuộc các item đó; {V("ml100k.link_yago_item_share")} item
  có liên kết YAGO.
- **Kiểm tra:** {V("link_check.n_wrong")}/{V("link_check.n_checked")} liên kết sai trong mẫu ngẫu nhiên
  ({V("link_check.n_auto")} phim xác nhận tự động bằng hai nguồn, {V("link_check.n_human")} phim kiểm tay).
  Khoảng tin cậy Wilson của tỉ lệ sai: [{V("link_check.wilson_low")}; {V("link_check.wilson_high")}]. Mẫu
  nhỏ nên chưa loại trừ được tỉ lệ sai đến {V("link_check.wilson_high")}.
- **Lỗi đã phát hiện:** KB4Rec ánh xạ "Clubland (1998)" sang *Cloudland Canyon State Park*.
"""
)

## 6. KG theo thời gian: fact có ngày, fact "tương lai", ví dụ Titanic
*Người trình bày: Triết*

Quy tắc (quyết định 0004 và 0006): KG dùng tại mốc *t* chỉ giữ fact đã đúng **trước** *t*. Ngày chỉ có năm được đọc là
31/12, ngày chỉ có tháng là ngày cuối tháng. KG cho train/val dựng tại t1, cho test dựng tại t2.

In [ ]:
say(
    f"""
- **Nguồn:** Wikidata, bản chụp {V("kg.snapshot")}: {V("kg.facts_total")} fact, {V("kg.relations")} quan hệ,
  {V("kg.films_with_facts")} phim có fact.
- **Fact có ngày rất ít:** chỉ {V("ml100k.statement_dated_share")} fact của ML-100K có ngày của chính
  statement; {V("kg.point_in_time_year_only_share")} giá trị `point_in_time` chỉ biết năm.
- **Fact "tương lai" bị loại tại t1** (ML-100K): {V("ml100k.dropped_future_tail_t1")} fact vì thực thể đích
  ra đời sau t1, {V("ml100k.dropped_future_statement_t1")} fact vì ngày của statement sau t1.
"""
)
rows = []
for variant, name in (
    ("default", "default (giữ fact không có ngày)"),
    ("drop_undated", "drop_undated"),
    ("static", "static (không lọc, chỉ để tham chiếu)"),
):
    rows.append(
        {
            "biến thể": name,
            "triple tại t1": V(f"ml100k.triples_{variant}_t1"),
            "triple tại t2": V(f"ml100k.triples_{variant}_t2"),
        }
    )
display(pd.DataFrame(rows))
figure("fig4_fact_bi_loai.png")

**Ví dụ Titanic.** Các giải thưởng của Titanic được trao sau t1 nhưng trước t2 của ML-100K, nên chúng chỉ được thấy ở KG của test.

In [ ]:
say(
    f"""
- {V("titanic.title")} có {V("titanic.facts")} fact; **{V("titanic.missing_t1")} fact bị loại tại t1**,
  trong đó {V("titanic.oscars_added_t2")} giải Oscar ngày {V("titanic.awards_date")} (t1 = {V("ml100k.t1")},
  t2 = {V("ml100k.t2")}). Dùng chúng cho rating trước ngày đó là rò rỉ tri thức tương lai: với KG theo mốc,
  chúng chỉ xuất hiện trong KG tại t2 (test).
- {V("titanic.dropped_both")} fact bị loại ở cả hai mốc, {V("titanic.dropped_both_year_only")} trong số đó
  chỉ biết năm (đọc là 31/12: thận trọng, có thể bỏ nhầm fact thật).
"""
)

## 7. Quy trình chia train/val/test và bằng chứng không rò rỉ
*Người trình bày: Triết*

Hai mốc chung t1 < t2, lấy theo phân vị thời gian của mọi timestamp (giá trị đọc từ cấu hình ở ô dưới). Rating ≥ 4 là
positive; k-core chỉ trên train; val/test chỉ giữ user và item đã có trong train; mask chỉ chứa tương tác trước mốc
(quyết định 0002).

In [ ]:
cfg = load_config(ROOT / "configs/data/ml100k.yaml")["split"]
say(
    f"Phân vị của t1 và t2: {str(cfg['q_val']).replace('.', ',')} và {str(cfg['q_test']).replace('.', ',')}; "
    f"k-core = {cfg['kcore']} (`configs/data/ml100k.yaml`)."
)
rows = []
for ds in ("ml100k", "ml1m"):
    rows.append(
        {
            "bộ dữ liệu": LABEL[ds],
            "t1": V(f"{ds}.t1"),
            "t2": V(f"{ds}.t2"),
            "positive train": V(f"{ds}.train_positives"),
            "user train": V(f"{ds}.train_users"),
            "positive val": V(f"{ds}.val_positives"),
            "user val": V(f"{ds}.val_users"),
            "positive test": V(f"{ds}.test_positives"),
            "user test": V(f"{ds}.test_users"),
        }
    )
display(pd.DataFrame(rows).set_index("bộ dữ liệu"))

**Sáu kiểm tra của `check_split`** (đọc từ `src/tkgat/data/leakage.py`). `scripts/02_build_splits.py` từ chối lưu split nếu
có bất kỳ vi phạm nào:

In [ ]:
checks = re.findall(r"^\s*# (\d)\. (.+)$", read_text("src/tkgat/data/leakage.py"), re.M)
say("\n".join(f"{i}. {text}" for i, text in checks))

**Kiểm tra trực tiếp trên split đã lưu.** Mỗi split chạy `check_split`; sau đó cố tình chèn một dòng test vào train, và một
dòng test vào mask, **chỉ trong bộ nhớ**, để thấy kiểm tra bắt được.

In [ ]:
def saved_split(run, config_file):
    cfg = load_config(ROOT / "configs/data" / config_file)
    cfg["processed_dir"] = str(ROOT / cfg["processed_dir"])
    return load_saved(cfg)[0]


if HAVE_DATA:
    splits = {"ML-100K": saved_split("ml-100k", "ml100k.yaml"), "ML-1M": saved_split("ml-1m", "ml1m.yaml")}
    say("\n".join(f"- {name}: `check_split` báo **{len(check_split(s))}** vấn đề." for name, s in splits.items()))
    s = splits["ML-100K"]
    broken = {
        "chèn một dòng test vào train": dataclasses.replace(
            s, train=pd.concat([s.train, s.test.head(1)], ignore_index=True)
        ),
        "chèn một dòng test vào mask của val": dataclasses.replace(
            s, seen_before_val=pd.concat([s.seen_before_val, s.test.head(1)[["user", "item", "ts"]]], ignore_index=True)
        ),
    }
    for what, bad in broken.items():
        problems = check_split(bad)
        say(f"- **{what}** (ML-100K, trong bộ nhớ): {len(problems)} vấn đề được phát hiện:")
        for p in problems:
            print("   ", p)
else:
    say("*chưa có: cần `data/processed/ml-100k` và `data/processed/ml-1m` (chạy `scripts/02_build_splits.py`).*")

**Test tự động và CI.**

In [ ]:
leak = [name for file, name in test_functions("test_") if file == "test_leakage.py"]
ci_steps = re.findall(r"- name: (.+)", read_text(".github/workflows/ci.yml"))
say(
    "- Test rò rỉ trong `tests/test_leakage.py`: "
    + ", ".join(f"`{n}`" for n in leak)
    + ". Các test `test_detects_*` cố tình "
    "chèn dữ liệu tương lai và yêu cầu `check_split` phải bắt được."
)
say("- CI (`.github/workflows/ci.yml`) chạy các bước: " + "; ".join(ci_steps) + ".")

## 8. Baseline trên 100K và 1M
*Người trình bày: Phôn*

Điểm trung bình ± độ lệch chuẩn mẫu (`ddof=1`) qua 5 seed; cấu hình chọn trên val, test chấm một lần. MostPop không ngẫu nhiên nên
chỉ có một lần chạy. *(Phôn bổ sung lời dẫn.)*

In [ ]:
rows = []
for ds in ("ml100k", "ml1m"):
    for model, name in MODEL_NAME.items():
        rows.append(
            {
                "bộ dữ liệu": LABEL[ds],
                "mô hình": name,
                "cấu hình chọn trên val": V(f"{ds}.{model}.selected"),
                "Recall@20 (test)": V(f"{ds}.{model}.test_recall20"),
                "NDCG@20 (test)": V(f"{ds}.{model}.test_ndcg20"),
                "số seed": V(f"{ds}.{model}.n_seeds"),
                "user test": V(f"{ds}.{model}.test_users"),
            }
        )
display(pd.DataFrame(rows).set_index(["bộ dữ liệu", "mô hình"]))
say("Số dòng test trong registry (`registry.load()`), để đối chiếu nguồn:")
display(REG[REG["phase"] == "test"].groupby(["dataset", "model"]).size().rename("số dòng test").to_frame())

**Kiểm định Wilcoxon theo từng user** (`results/significance.csv`, do `scripts/09_significance.py` ghi; Recall@20, hiệu chỉnh Holm
trong cùng bộ dữ liệu và metric). `n_nonzero` là số user có chênh lệch khác 0, tức cỡ mẫu hiệu dụng của kiểm định.

In [ ]:
sig_path = ROOT / "results/significance.csv"
if sig_path.exists():
    sig = pd.read_csv(sig_path)
    sig = sig[sig["metric"] == "recall@20"]
    cols = ["dataset", "model_a", "model_b", "n_pairs", "n_nonzero", "mean_diff", "p_raw", "p_holm", "warning"]
    display(
        sig[cols]
        .fillna("")
        .style.format({"mean_diff": "{:.4f}", "p_raw": "{:.3g}", "p_holm": "{:.3g}"})
        .hide(axis="index")
    )
    for _, r in sig[sig["warning"].notna()].drop_duplicates("dataset").iterrows():
        say(f"> **Cảnh báo ({r['dataset']}):** {r['warning']}.")
else:
    say("*chưa có: `results/significance.csv` (chạy `python scripts/09_significance.py`).*")

**Độ nhạy theo cách chia** (chỉ val; điểm val tuyệt đối không so sánh được giữa các split, chỉ so thứ tự ba mô hình):

In [ ]:
sens = ROOT / "docs/sensitivity-val.md"
if sens.exists():
    say("\n".join(read_text("docs/sensitivity-val.md").splitlines()[1:]))
else:
    say("*chưa có: `docs/sensitivity-val.md` (chạy `python scripts/10_sensitivity.py`).*")

**Learning curve của BPR-MF** (loss huấn luyện và Recall@20 trên val theo epoch, trung bình ± std qua 5 seed; nét liền chỉ vẽ khi
đủ cả 5 seed) và **tham số huấn luyện**:

In [ ]:
figure("fig5_bprmf_curve_ml-100k.png")
figure("fig5_bprmf_curve_ml-1m.png")
table = ROOT / "docs/bprmf-training-table.md"
say(
    "\n".join(read_text("docs/bprmf-training-table.md").splitlines()[2:])
    if table.exists()
    else "*chưa có: `docs/bprmf-training-table.md`.*"
)

## 9. Repo và cách làm việc mới
*Người trình bày: Phôn*

*(Phôn bổ sung lời dẫn: một nhánh một PR, CI chạy ruff và pytest, quyết định ghi thành file `docs/decisions/`, registry chỉ thêm
dòng, dữ liệu không vào git.)* Các số dưới đây được đếm từ chính repo:

In [ ]:
all_tests = test_functions("test_")
decisions = sorted((ROOT / "docs/decisions").glob("0*.md"))
scripts = sorted(p.name for p in (ROOT / "scripts").glob("*.py"))
say(
    f"""
- **Test:** {len(all_tests)} hàm test trong {len({file for file, _ in all_tests})} file (`tests/`).
- **Quyết định đã ghi:** {len(decisions)} file trong `docs/decisions/`.
- **Script:** {len(scripts)} file trong `scripts/`: {", ".join(f"`{s}`" for s in scripts)}.
- **Registry:** `results/registry.csv` chỉ được thêm dòng; đọc qua `tkgat.registry.load()` (dòng mới nhất
  mỗi khóa).
"""
)

## 10. Kế hoạch từ 02/11 và các câu hỏi xin ý kiến cô
*Người trình bày: Cả hai (Phôn và Triết)*

**Quyết định và trạng thái.** Kế hoạch từ 02/11 bám paper KGAT (Wang và cộng sự, KDD 2019) và ba hướng cô gợi ý ngày 18/09
(rating như relation hoặc trọng số, timestamp như relation, LLM làm sạch metadata phim). Bảng dưới lấy tiêu đề và dòng `Status`
nguyên văn của từng file trong `docs/decisions/`: trạng thái của các quyết định được tính từ repo, không gõ tay.

In [ ]:
rows = []
for path in sorted((ROOT / "docs/decisions").glob("0*.md")):
    lines = path.read_text(encoding="utf-8").splitlines()
    status = next((x for x in lines if x.startswith("- Status:")), "- Status: chưa ghi")
    rows.append(
        {
            "quyết định": path.stem,
            "tiêu đề": lines[0].lstrip("# ").strip(),
            "Status (nguyên văn)": status.removeprefix("- Status:").strip(),
        }
    )
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(rows).style.hide(axis="index"))

> Kế hoạch đang dùng nằm ở artifact: https://claude.ai/artifact/5KUTXrnhN5ZA8gR2XJNjXs. `docs/plan.md` là bản gốc ngày 17/10, chỉ giữ để đối chiếu.

**Các câu hỏi xin ý kiến cô** (đọc từ `docs/data-report.md`, mục 8):

In [ ]:
report = read_text("docs/data-report.md")
block = re.search(r"\*\*Câu hỏi xin ý kiến cô\*\*\n(.*?)\n\n\*\*", report, re.S)
say(block.group(1) if block else "*chưa có: không tìm thấy mục câu hỏi trong `docs/data-report.md`.*")

**Kết quả trao đổi về 4 nội dung báo cô ngày 19/10:** *chưa có: kết quả trao đổi không được ghi trong repo; nhóm bổ sung từ
biên bản trước buổi present.*

## 11. KGAT và T-KGAT (sơ bộ)
*Người trình bày: Triết*

> **Kết quả sơ bộ, chưa đóng băng (tag `v0.2-kgat`), chạy trước khi cô duyệt quy trình.** `ml-1m-u10` là 10% user của ML-1M,
> không phải ML-1M đầy đủ (quyết định 0008).

Bảng, kiểm định và hình dưới đây do `scripts/10_rq3_table.py` ghi (`results/rq3_models.csv`, `results/rq3_pairs.csv`,
`docs/figures/rq3_curves_*.png`); notebook chỉ đọc chúng và không chạy lại gì. Chỗ nào chưa có file thì ghi "chưa chạy".

In [ ]:
from tkgat.analysis.report import vn_p, vn_pm

RQ3_MODELS, RQ3_PAIRS = ROOT / "results/rq3_models.csv", ROOT / "results/rq3_pairs.csv"
SUBSET_RUNS = {"ml-1m-u10", "ml-1m-u05"}  # random user subsets of ML-1M (decision 0008)
PRELIMINARY = {"kgat", "kgat_nokg", "tkgat", "tkgat_perm"}
NOT_RUN = "chưa chạy"


def read_rq3(path):
    # the file written by scripts/10_rq3_table.py, or None while it has not been written (or has no row)
    try:
        return pd.read_csv(path) if path.exists() else None
    except pd.errors.EmptyDataError:
        return None


def number(value, digits=4):
    return f"{value:.{digits}f}".replace(".", ",")


def sample_std(row, metric):
    # one seed has no sample std: the CSV holds an empty cell
    return None if pd.isna(row[f"{metric}_std"]) else row[f"{metric}_std"]


kgat_rows = REG[REG["model"].str.contains("kgat", case=False)]
say(f"Số dòng KGAT/T-KGAT trong registry: **{len(kgat_rows)}**.")

models = read_rq3(RQ3_MODELS)
if models is None:
    say(
        f"*{NOT_RUN}: chưa có `results/rq3_models.csv` "
        "(chạy `scripts/10_rq3_table.py` khi KGAT, T-KGAT, đối chứng đã chạy).*"
    )
else:
    for dataset, part in models.groupby("dataset", sort=False):
        say(f"**{dataset}**" + (" (bộ con của ML-1M, không phải ML-1M đầy đủ)" if dataset in SUBSET_RUNS else ""))
        rows = []
        for _, r in part.iterrows():
            ok = r["status"] == "ok"
            rows.append(
                {
                    "mô hình": r["label"] + (" [sơ bộ]" if r["model"] in PRELIMINARY else ""),
                    "cấu hình": r["point"] if ok else "—",
                    "Recall@20 (test)": vn_pm(r["recall@20_mean"], sample_std(r, "recall@20")) if ok else NOT_RUN,
                    "NDCG@20 (test)": vn_pm(r["ndcg@20_mean"], sample_std(r, "ndcg@20")) if ok else NOT_RUN,
                    "số seed": int(r["n_seeds"]) if ok else "—",
                }
            )
        display(pd.DataFrame(rows).style.hide(axis="index"))

pairs = read_rq3(RQ3_PAIRS)
if pairs is None:
    say(f"*{NOT_RUN}: chưa có `results/rq3_pairs.csv`, nên chưa có kiểm định cặp theo user.*")
else:
    rows = []
    for _, r in pairs.iterrows():
        ok = r["status"] == "ok"
        note = r.get("warning") if ok else r["status"]
        rows.append(
            {
                "bộ dữ liệu": r["dataset"],
                "so sánh": r["pair"],
                "metric": r["metric"],
                "n_pairs": int(r["n_pairs"]) if ok else "—",
                "n_nonzero": int(r["n_nonzero"]) if ok else "—",
                "hiệu trung bình": number(r["mean_diff"]) if ok else "—",
                "p thô": vn_p(r["p_raw"]) if ok else "—",
                "p Holm": vn_p(r["p_holm"]) if ok else "—",
                "ghi chú": note if isinstance(note, str) else "",
            }
        )
    display(pd.DataFrame(rows).style.hide(axis="index"))

curves = sorted((ROOT / "docs/figures").glob("rq3_curves_*.png"))
for path in curves:
    say(f"*Learning curve val, {path.stem.removeprefix('rq3_curves_')}:*")
    display(Image(filename=path, width=720))
if not curves:
    say(f"*{NOT_RUN}: chưa có hình `docs/figures/rq3_curves_*.png` (cần history của KGAT, T-KGAT, T-KGAT hoán vị).*")